<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/Stress_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [ ]:
# --- CONFIGURATION ---
DATASET_PATH = 'WESAD/'  # Apne unzip folder ka path dein
SUBJECTS = ['S2', 'S3', 'S4', 'S5']  # Pehle 4 subjects testing ke liye
TARGET_SAMPLING_RATE = 4  # Resampling target (4Hz)
WINDOW_SIZE = 60  # 60 seconds ka sliding window
STEP_SIZE = 10  # 10 seconds ka step (overlap ke liye)

def resample_signal(raw_signal, original_fs, target_fs):
    """
    Signals ko resample karne ka function taake dono (EDA aur PPG)
    ki length aur sampling rate match ho sake.
    """
    num_samples = int(len(raw_signal) * (target_fs / original_fs))
    resampled = signal.resample(raw_signal, num_samples)
    return resampled.flatten()

def create_sliding_windows(data_matrix, labels, window_size_samples, step_size_samples):
    """
    Continuous signals ko 1D-CNN aur Transformer ke liye 3D array (Windows) mein convert karta hai.
    """
    X, y = [], []
    for i in range(0, len(data_matrix) - window_size_samples, step_size_samples):
        # Window extract karein
        window = data_matrix[i : i + window_size_samples, :]
        # Us window ke labels ka mode (jo label sab se zyada baar aya) pick karein
        window_labels = labels[i : i + window_size_samples]

        # Sirf valid labels ko rakhein (1: Baseline, 2: Stress, 3: Amusement)
        # Baki codes (0, 4, etc.) jo transition/rest hain unhein nikal dein
        unique_labels, counts = np.unique(window_labels, return_counts=True)
        majority_label = unique_labels[np.argmax(counts)]

        if majority_label in:
            X.append(window)
            y.append(majority_label)

    return np.array(X), np.array(y)

# --- MAIN PREPROCESSING PIPELINE ---
all_features = []
all_labels = []

print("⏳ Data processing shuru ho raha hai...")

for sub in SUBJECTS:
    file_path = os.path.join(DATASET_PATH, sub, f"{sub}.pkl")
    if not os.path.exists(file_path):
        print(f"⚠️ Error: {file_path} nahi mili. Path check karein.")
        continue

    print(f"📦 Loading Subject: {sub}...")
    with open(file_path, 'rb') as f:
        # latin1 encoding WESAD pickle files ke liye zaroori hai
        data = pickle.load(f, encoding='latin1')

    # 1. Raw Signals aur Labels Extract karein
    eda_raw = data['signal']['wrist']['EDA'].flatten()  # Original FS: 4Hz
    bvp_raw = data['signal']['wrist']['BVP'].flatten()  # Original FS: 64Hz (PPG)
    labels_raw = data['label'].flatten()                # Original FS: 700Hz (Chest RespiBAN base)

    # 2. Sampling Rates Match Karein (Resampling)
    # EDA pehle se 4Hz par hai, isay resample karne ki zaroorat nahi
    eda_resampled = eda_raw

    # BVP ko 64Hz se 4Hz par le kar aana hai
    bvp_resampled = resample_signal(bvp_raw, original_fs=64, target_fs=TARGET_SAMPLING_RATE)

    # Labels 700Hz par hain, unhein bhi 4Hz par le kar aana hai (Nearest Neighbor method)
    label_indices = np.linspace(0, len(labels_raw) - 1, len(eda_resampled), dtype=int)
    labels_resampled = labels_raw[label_indices]

    # 3. Dono signals ki length check aur trim karein (Taake match hon)
    min_len = min(len(eda_resampled), len(bvp_resampled), len(labels_resampled))
    eda_resampled = eda_resampled[:min_len]
    bvp_resampled = bvp_resampled[:min_len]
    labels_resampled = labels_resampled[:min_len]

    # 4. Features ko scale/normalize karein (1D-CNN ke liye zaroori hai)
    scaler = StandardScaler()
    combined_features = np.column_stack((eda_resampled, bvp_resampled))
    scaled_features = scaler.fit_transform(combined_features)

    # 5. Sliding Windows Banayein
    window_samples = WINDOW_SIZE * TARGET_SAMPLING_RATE  # 60 sec * 4Hz = 240 samples
    step_samples = STEP_SIZE * TARGET_SAMPLING_RATE      # 10 sec * 4Hz = 40 samples

    X_sub, y_sub = create_sliding_windows(scaled_features, labels_resampled, window_samples, step_samples)

    all_features.append(X_sub)
    all_labels.append(y_sub)
    print(f"✅ Subject {sub} successfully processed! Windows count: {len(X_sub)}")

# --- FINAL DATA CONCATENATION ---
X_final = np.concatenate(all_features, axis=0)
y_final = np.concatenate(all_labels, axis=0)

# Binary Classification ke liye labels adjust karein (Optional)
# 1: Baseline (Non-Stress), 2: Stress, 3: Amusement (Non-Stress)
# Agar aap Binary karna chahein toh: y_final = np.where(y_final == 2, 1, 0)

print("\n🚀 --- PREPROCESSING COMPLETED ---")
print(f"Final Input Shape (X): {X_final.shape} -> (Windows, Time-steps/Samples, Features)")
print(f"Final Labels Shape (y): {y_final.shape}")
